In [1]:
import pandas as pd

In [2]:
tickets = pd.read_csv("../data/tickets.csv")
tickets.head()

,ticket_id,deal_id,created_date,updated_date,priority,severity,status,issue_category,assigned_team,assigned_engineer,blocker_flag,customer_impact,sla_breach,estimated_resolution_days,ticket_summary,ticket_description
0,TICK-9001,DEAL-1054,2026-03-23,2026-03-30,High,Sev-3,In Progress,Security Review,Implementation,John Hicks,Yes,Medium,Yes,6,Pending security approvals blocking deployment,"Security review approvals remain pending, impa..."
1,TICK-9002,DEAL-1119,2026-01-04,2026-01-08,High,Sev-2,Open,Reporting Mismatch,Data Engineering,Jennifer Kerr,Yes,Low,No,8,Reporting discrepancies found during validation,Customer reports do not match expected output ...
2,TICK-9003,DEAL-1120,2026-03-21,2026-03-26,Low,Sev-4,Open,Client Dependency,Data Engineering,Dale Martin,No,Medium,No,6,Customer-side dependency blocking milestone co...,Customer stakeholders have not completed requi...
3,TICK-9004,DEAL-1082,2026-01-24,2026-02-01,High,Sev-1,Pending,Data Migration,Implementation,Catherine Byrd,Yes,High,Yes,9,Historical data migration validation failures ...,Migration validation revealed missing historic...
4,TICK-9005,DEAL-1193,2026-02-19,2026-02-25,Medium,Sev-2,Open,Client Dependency,Support,Gerald Reese,Yes,Low,No,1,Customer-side dependency blocking milestone co...,Customer stakeholders have not completed requi...


In [3]:
tickets["combined_text"] = (
    "Summary: " + tickets["ticket_summary"].astype(str) +
    "\nDescription: " + tickets["ticket_description"].astype(str) +
    "\nPriority: " + tickets["priority"].astype(str) +
    "\nStatus: " + tickets["status"].astype(str) +
    "\nCategory: " + tickets["issue_category"].astype(str)
)

tickets["combined_text"].iloc[0]

'Summary: Pending security approvals blocking deployment\nDescription: Security review approvals remain pending, impacting deployment timelines.\nPriority: High\nStatus: In Progress\nCategory: Security Review'

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sample_embedding = model.encode(
    tickets["combined_text"].iloc[0]
)

len(sample_embedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384

In [5]:
len(sample_embedding)

384

In [6]:
ticket_texts = tickets["combined_text"].tolist()

embeddings = model.encode(ticket_texts)

len(embeddings)

5000

In [7]:
import chromadb

In [8]:
client = chromadb.Client()

In [9]:
collection = client.create_collection(
    name="enterprise_tickets"
)

In [10]:
collection.add(
    embeddings=embeddings.tolist(),
    documents=tickets["combined_text"].tolist(),
    ids=tickets["ticket_id"].tolist()
)

In [11]:
results = collection.query(
    query_texts=[
        "critical API integration blockers"
    ],
    n_results=5
)

results

{'ids': [['TICK-11374',
   'TICK-11378',
   'TICK-11628',
   'TICK-12986',
   'TICK-13946']],
 'embeddings': None,
 'documents': [['Summary: Customer-side dependency blocking milestone completion\nDescription: Customer stakeholders have not completed required approvals and access provisioning.\nPriority: Critical\nStatus: Open\nCategory: Client Dependency',
   'Summary: Customer-side dependency blocking milestone completion\nDescription: Customer stakeholders have not completed required approvals and access provisioning.\nPriority: Critical\nStatus: Open\nCategory: Client Dependency',
   'Summary: Customer-side dependency blocking milestone completion\nDescription: Customer stakeholders have not completed required approvals and access provisioning.\nPriority: Critical\nStatus: Open\nCategory: Client Dependency',
   'Summary: Customer-side dependency blocking milestone completion\nDescription: Customer stakeholders have not completed required approvals and access provisioning.\nPriority

In [12]:
for i, doc in enumerate(results["documents"][0]):
    print(f"\nRESULT {i+1}")
    print("-" * 50)
    # Print the ID of the matching document
    print(f"ID: {results['ids'][0][i]}")
    print(doc)


RESULT 1
--------------------------------------------------
ID: TICK-11374
Summary: Customer-side dependency blocking milestone completion
Description: Customer stakeholders have not completed required approvals and access provisioning.
Priority: Critical
Status: Open
Category: Client Dependency

RESULT 2
--------------------------------------------------
ID: TICK-11378
Summary: Customer-side dependency blocking milestone completion
Description: Customer stakeholders have not completed required approvals and access provisioning.
Priority: Critical
Status: Open
Category: Client Dependency

RESULT 3
--------------------------------------------------
ID: TICK-11628
Summary: Customer-side dependency blocking milestone completion
Description: Customer stakeholders have not completed required approvals and access provisioning.
Priority: Critical
Status: Open
Category: Client Dependency

RESULT 4
--------------------------------------------------
ID: TICK-12986
Summary: Customer-side depende

In [13]:
results = collection.query(
    query_texts=[
        "projects with security delays"
    ],
    n_results=5
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nRESULT {i+1}")
    print("-" * 50)
    # Print the ID of the matching document
    print(f"ID: {results['ids'][0][i]}")
    print(doc)


RESULT 1
--------------------------------------------------
ID: TICK-10157
Summary: Pending security approvals blocking deployment
Description: Security review approvals remain pending, impacting deployment timelines.
Priority: Critical
Status: In Progress
Category: Security Review

RESULT 2
--------------------------------------------------
ID: TICK-10662
Summary: Pending security approvals blocking deployment
Description: Security review approvals remain pending, impacting deployment timelines.
Priority: Critical
Status: In Progress
Category: Security Review

RESULT 3
--------------------------------------------------
ID: TICK-11811
Summary: Pending security approvals blocking deployment
Description: Security review approvals remain pending, impacting deployment timelines.
Priority: Critical
Status: In Progress
Category: Security Review

RESULT 4
--------------------------------------------------
ID: TICK-12373
Summary: Pending security approvals blocking deployment
Description: Sec

In [14]:
results = collection.query(
    query_texts=[
        "customer escalation risk"
    ],
    n_results=5
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nRESULT {i+1}")
    print("-" * 50)
    # Print the ID of the matching document
    print(f"ID: {results['ids'][0][i]}")
    print(doc)


RESULT 1
--------------------------------------------------
ID: TICK-10583
Summary: Customer environment provisioning delayed
Description: Infrastructure provisioning delays are preventing integration and testing activities.
Priority: Critical
Status: Open
Category: Environment Provisioning

RESULT 2
--------------------------------------------------
ID: TICK-12666
Summary: Customer environment provisioning delayed
Description: Infrastructure provisioning delays are preventing integration and testing activities.
Priority: Critical
Status: Open
Category: Environment Provisioning

RESULT 3
--------------------------------------------------
ID: TICK-13116
Summary: Customer environment provisioning delayed
Description: Infrastructure provisioning delays are preventing integration and testing activities.
Priority: Critical
Status: Open
Category: Environment Provisioning

RESULT 4
--------------------------------------------------
ID: TICK-13183
Summary: Customer environment provisioning de

In [15]:
results = collection.query(
    query_texts=[
        "workflow automation failures"
    ],
    n_results=5
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nRESULT {i+1}")
    print("-" * 50)
    # Print the ID of the matching document
    print(f"ID: {results['ids'][0][i]}")
    print(doc)


RESULT 1
--------------------------------------------------
ID: TICK-9027
Summary: Workflow automation execution failing intermittently
Description: Automated workflows are failing intermittently due to orchestration dependency issues.
Priority: Critical
Status: Resolved
Category: Workflow Failure

RESULT 2
--------------------------------------------------
ID: TICK-10541
Summary: Workflow automation execution failing intermittently
Description: Automated workflows are failing intermittently due to orchestration dependency issues.
Priority: Critical
Status: Resolved
Category: Workflow Failure

RESULT 3
--------------------------------------------------
ID: TICK-11371
Summary: Workflow automation execution failing intermittently
Description: Automated workflows are failing intermittently due to orchestration dependency issues.
Priority: Critical
Status: Resolved
Category: Workflow Failure

RESULT 4
--------------------------------------------------
ID: TICK-11652
Summary: Workflow auto

In [16]:
results = collection.query(
    query_texts=[
        "tickets causing go-live delays"
    ],
    n_results=5
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nRESULT {i+1}")
    print("-" * 50)
    # Print the ID of the matching document
    print(f"ID: {results['ids'][0][i]}")
    print(doc)


RESULT 1
--------------------------------------------------
ID: TICK-10310
Summary: Customer environment provisioning delayed
Description: Infrastructure provisioning delays are preventing integration and testing activities.
Priority: High
Status: Resolved
Category: Environment Provisioning

RESULT 2
--------------------------------------------------
ID: TICK-10387
Summary: Customer environment provisioning delayed
Description: Infrastructure provisioning delays are preventing integration and testing activities.
Priority: High
Status: Resolved
Category: Environment Provisioning

RESULT 3
--------------------------------------------------
ID: TICK-11959
Summary: Customer environment provisioning delayed
Description: Infrastructure provisioning delays are preventing integration and testing activities.
Priority: High
Status: Resolved
Category: Environment Provisioning

RESULT 4
--------------------------------------------------
ID: TICK-12206
Summary: Customer environment provisioning de

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")


print(api_key[:10])

sk-proj-f9


In [18]:
from openai import OpenAI

client_openai = OpenAI(api_key=api_key)

In [19]:
query = "What are the biggest API integration risks?"

results = collection.query(
    query_texts=[query],
    n_results=5
)

retrieved_docs = results["documents"][0]

retrieved_docs

['Summary: API synchronization failing during nightly processing\nDescription: API jobs are intermittently failing due to token validation and timeout handling issues.\nPriority: Critical\nStatus: In Progress\nCategory: API Integration',
 'Summary: API synchronization failing during nightly processing\nDescription: API jobs are intermittently failing due to token validation and timeout handling issues.\nPriority: Critical\nStatus: In Progress\nCategory: API Integration',
 'Summary: API synchronization failing during nightly processing\nDescription: API jobs are intermittently failing due to token validation and timeout handling issues.\nPriority: Critical\nStatus: In Progress\nCategory: API Integration',
 'Summary: API synchronization failing during nightly processing\nDescription: API jobs are intermittently failing due to token validation and timeout handling issues.\nPriority: Critical\nStatus: In Progress\nCategory: API Integration',
 'Summary: API synchronization failing during ni

In [20]:
context = "\n\n".join(retrieved_docs)

print(context[:2000])

Summary: API synchronization failing during nightly processing
Description: API jobs are intermittently failing due to token validation and timeout handling issues.
Priority: Critical
Status: In Progress
Category: API Integration

Summary: API synchronization failing during nightly processing
Description: API jobs are intermittently failing due to token validation and timeout handling issues.
Priority: Critical
Status: In Progress
Category: API Integration

Summary: API synchronization failing during nightly processing
Description: API jobs are intermittently failing due to token validation and timeout handling issues.
Priority: Critical
Status: In Progress
Category: API Integration

Summary: API synchronization failing during nightly processing
Description: API jobs are intermittently failing due to token validation and timeout handling issues.
Priority: Critical
Status: In Progress
Category: API Integration

Summary: API synchronization failing during nightly processing
Description: 

In [21]:
prompt = f"""
You are an enterprise implementation executive assistant.

Use the provided enterprise ticket data to answer the question.

QUESTION:
{query}

TICKET DATA:
{context}

Provide:
1. Executive summary
2. Key risks
3. Operational concerns
4. Recommended next steps
"""

In [22]:
response = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "You are an enterprise implementation intelligence assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3
)

answer = response.choices[0].message.content

print(answer)

1. Executive Summary:
The primary risk facing the API integration is the intermittent failure of API synchronization jobs during nightly processing. These failures are critically impacting operations due to issues with token validation and timeout handling. The problem is ongoing and currently being addressed, but it poses a significant threat to system reliability and data consistency.

2. Key Risks:
- Token validation failures causing authentication breakdowns.
- Timeout handling issues leading to incomplete or failed API jobs.
- Repeated failures during critical nightly synchronization windows.
- Potential data inconsistency or loss due to failed API processes.
- Operational disruption impacting downstream systems dependent on API data.

3. Operational Concerns:
- Nightly processing delays or failures could cascade into business process delays.
- Increased manual intervention may be required to address failed jobs.
- Risk of degraded system performance or outages if issues persist.


In [23]:
meeting_notes = pd.read_csv("../data/meeting_notes.csv")
status_reports = pd.read_csv("../data/status_reports.csv")
escalations = pd.read_csv("../data/escalations.csv")
risks = pd.read_csv("../data/risks.csv")

deals_df = pd.read_csv("../data/deals.csv")
#tickets = pd.read_csv("../data/tickets.csv")
#escalations = pd.read_csv("../data/escalations.csv")
#risks = pd.read_csv("../data/risks.csv")
#meeting_notes = pd.read_csv("../data/meeting_notes.csv")
#status_reports = pd.read_csv("../data/status_reports.csv")

In [24]:
meeting_notes["combined_text"] = (
    "Meeting Type: " + meeting_notes["meeting_type"].astype(str) +
    "\nMeeting Notes: " + meeting_notes["meeting_notes"].astype(str) +
    "\nAction Items: " + meeting_notes["action_items"].astype(str)
)

meeting_notes["combined_text"].iloc[0]

'Meeting Type: Technical Deep Dive\nMeeting Notes: Client stakeholders requested revised go-live timeline following environment instability.\nAction Items: Implementation team to finalize deployment checklist.'

In [25]:
status_reports["combined_text"] = (
    "Executive Summary: " + status_reports["executive_summary"].astype(str) +
    "\nCurrent Risks: " + status_reports["current_risks"].astype(str) +
    "\nNext Steps: " + status_reports["next_steps"].astype(str)
)

status_reports["combined_text"].iloc[0]

'Executive Summary: Go-live timeline remains at risk due to unresolved dependencies.\nCurrent Risks: Delayed customer approvals impacting timeline.\nNext Steps: Resolve open dependency blockers.'

In [26]:
escalations["combined_text"] = (
    "Escalation Reason: " + escalations["escalation_reason"].astype(str) +
    "\nEscalation Notes: " + escalations["escalation_notes"].astype(str)
)

escalations["combined_text"].iloc[0]

'Escalation Reason: Repeated integration failures\nEscalation Notes: Executive leadership requested immediate remediation plan.'

In [27]:
risks["combined_text"] = (
    "Risk Category: " + risks["risk_category"].astype(str) +
    "\nMitigation Plan: " + risks["mitigation_plan"].astype(str) +
    "\nStatus: " + risks["current_status"].astype(str)
)

risks["combined_text"].iloc[0]

'Risk Category: Testing Instability\nMitigation Plan: Weekly executive escalation meetings.\nStatus: Monitoring'

In [28]:
enterprise_collection = client.create_collection(
    name="enterprise_knowledge"
)

In [29]:
meeting_embeddings = model.encode(
    meeting_notes["combined_text"].tolist()
)

enterprise_collection.add(
    embeddings=meeting_embeddings.tolist(),
    documents=meeting_notes["combined_text"].tolist(),
    ids=meeting_notes["meeting_id"].tolist()
)

In [30]:
status_embeddings = model.encode(
    status_reports["combined_text"].tolist()
)

enterprise_collection.add(
    embeddings=status_embeddings.tolist(),
    documents=status_reports["combined_text"].tolist(),
    ids=status_reports["report_id"].tolist()
)

In [31]:
escalation_embeddings = model.encode(
    escalations["combined_text"].tolist()
)

enterprise_collection.add(
    embeddings=escalation_embeddings.tolist(),
    documents=escalations["combined_text"].tolist(),
    ids=escalations["escalation_id"].tolist()
)

In [32]:
risk_embeddings = model.encode(
    risks["combined_text"].tolist()
)

enterprise_collection.add(
    embeddings=risk_embeddings.tolist(),
    documents=risks["combined_text"].tolist(),
    ids=risks["risk_id"].tolist()
)

In [33]:
tickets.columns

Index(['ticket_id', 'deal_id', 'created_date', 'updated_date', 'priority',
       'severity', 'status', 'issue_category', 'assigned_team',
       'assigned_engineer', 'blocker_flag', 'customer_impact', 'sla_breach',
       'estimated_resolution_days', 'ticket_summary', 'ticket_description',
       'combined_text'],
      dtype='str')

In [34]:
enterprise_collection.add(
    embeddings=embeddings.tolist(),
    documents=tickets["combined_text"].tolist(),
    ids=tickets["ticket_id"].tolist()
)

In [35]:
query = "What implementation risks are threatening customer go-live timelines?"

results = enterprise_collection.query(
    query_texts=[query],
    n_results=10
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nRESULT {i+1}")
    print("-" * 50)
    print(doc)


RESULT 1
--------------------------------------------------
Executive Summary: Go-live timeline remains at risk due to unresolved dependencies.
Current Risks: Delayed customer approvals impacting timeline.
Next Steps: Resolve open dependency blockers.

RESULT 2
--------------------------------------------------
Executive Summary: Go-live timeline remains at risk due to unresolved dependencies.
Current Risks: Delayed customer approvals impacting timeline.
Next Steps: Resolve open dependency blockers.

RESULT 3
--------------------------------------------------
Executive Summary: Go-live timeline remains at risk due to unresolved dependencies.
Current Risks: Delayed customer approvals impacting timeline.
Next Steps: Resolve open dependency blockers.

RESULT 4
--------------------------------------------------
Executive Summary: Go-live timeline remains at risk due to unresolved dependencies.
Current Risks: Delayed customer approvals impacting timeline.
Next Steps: Resolve open dependenc

In [36]:
context = "\n\n".join(results["documents"][0])

prompt = f"""
You are an executive implementation intelligence assistant.

Use the enterprise data below to answer the question.

QUESTION:
{query}

ENTERPRISE DATA:
{context}

Provide:
1. Executive Summary
2. Key Risks
3. Escalation Concerns
4. Operational Blockers
5. Recommended Executive Actions
"""

In [37]:
response = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "You are an enterprise implementation intelligence assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3
)

answer = response.choices[0].message.content

print(answer)

1. Executive Summary  
The customer go-live timeline is currently at significant risk due to multiple unresolved dependencies. The primary factor threatening the schedule is the delay in receiving necessary customer approvals, which is causing a bottleneck in progressing critical implementation tasks.

2. Key Risks  
- Delayed customer approvals leading to stalled decision-making.  
- Unresolved dependency blockers that prevent advancement to subsequent implementation phases.  
- Potential cascading delays impacting overall project milestones and deliverables.

3. Escalation Concerns  
- Continued delays in customer approvals may require executive-level intervention to expedite decision processes.  
- Risk of contractual or service level agreement breaches if go-live timelines are not met.  
- Possible erosion of customer confidence and satisfaction due to timeline slippage.

4. Operational Blockers  
- Open dependency blockers that are contingent on customer sign-offs.  
- Lack of ali

In [38]:
ticket_counts = (
    tickets.groupby("deal_id")
    .size()
    .reset_index(name="ticket_count")
)

ticket_counts.head()

,deal_id,ticket_count
0,DEAL-1001,28
1,DEAL-1002,12
2,DEAL-1003,45
3,DEAL-1004,22
4,DEAL-1005,11


In [39]:
escalation_counts = (
    escalations.groupby("deal_id")
    .size()
    .reset_index(name="escalation_count")
)

escalation_counts.head()

,deal_id,escalation_count
0,DEAL-1003,4
1,DEAL-1006,6
2,DEAL-1012,10
3,DEAL-1013,7
4,DEAL-1019,9


In [40]:
risk_counts = (
    risks.groupby("deal_id")
    .size()
    .reset_index(name="risk_count")
)

risk_counts.head()

,deal_id,risk_count
0,DEAL-1001,2
1,DEAL-1002,5
2,DEAL-1003,3
3,DEAL-1004,5
4,DEAL-1005,8


In [41]:
project_risk_df = deals_df.copy()

project_risk_df = project_risk_df.merge(
    ticket_counts,
    on="deal_id",
    how="left"
)

project_risk_df = project_risk_df.merge(
    escalation_counts,
    on="deal_id",
    how="left"
)

project_risk_df = project_risk_df.merge(
    risk_counts,
    on="deal_id",
    how="left"
)

project_risk_df.head()

,deal_id,customer_id,project_name,project_type,project_status,health_profile,implementation_phase,kickoff_date,planned_go_live,estimated_delay_days,project_manager,technical_lead,health_score,executive_visibility,open_blockers,budget_utilization_pct,ticket_count,escalation_count,risk_count
0,DEAL-1001,CUST-004,Brown-Porter API Modernization,ERP Integration,Yellow,At-Risk,Discovery,2026-02-12,2026-11-08,14,Jennifer Fox,Nicholas Cochran,78,Critical,3,72,28,NaN,2.0
1,DEAL-1002,CUST-034,Crosby-Rowland ERP Integration,CRM Implementation,Green,Healthy,Discovery,2025-12-02,2026-09-20,4,David Castillo,Cindy Benton,85,Medium,0,65,12,NaN,5.0
2,DEAL-1003,CUST-008,Sanchez LLC CRM Implementation,ERP Integration,Red,Critical,UAT,2026-02-16,2026-12-01,57,Megan Sandoval,Kelly Collins,22,High,11,56,45,4.0,3.0
3,DEAL-1004,CUST-014,Wilson Inc Data Migration,Cloud Migration,Yellow,At-Risk,Testing,2026-02-24,2026-10-19,11,Mary Buckley,Ashley Peterson,63,High,4,88,22,NaN,5.0
4,DEAL-1005,CUST-005,Hamilton LLC ERP Integration,CRM Implementation,Green,Healthy,Build,2026-03-14,2026-11-18,4,Carolyn Carter,Kevin Ingram,83,High,2,48,11,NaN,8.0


In [43]:
project_risk_df = project_risk_df.fillna(0)

In [44]:
project_risk_df.head()

,deal_id,customer_id,project_name,project_type,project_status,health_profile,implementation_phase,kickoff_date,planned_go_live,estimated_delay_days,project_manager,technical_lead,health_score,executive_visibility,open_blockers,budget_utilization_pct,ticket_count,escalation_count,risk_count
0,DEAL-1001,CUST-004,Brown-Porter API Modernization,ERP Integration,Yellow,At-Risk,Discovery,2026-02-12,2026-11-08,14,Jennifer Fox,Nicholas Cochran,78,Critical,3,72,28,0.0,2.0
1,DEAL-1002,CUST-034,Crosby-Rowland ERP Integration,CRM Implementation,Green,Healthy,Discovery,2025-12-02,2026-09-20,4,David Castillo,Cindy Benton,85,Medium,0,65,12,0.0,5.0
2,DEAL-1003,CUST-008,Sanchez LLC CRM Implementation,ERP Integration,Red,Critical,UAT,2026-02-16,2026-12-01,57,Megan Sandoval,Kelly Collins,22,High,11,56,45,4.0,3.0
3,DEAL-1004,CUST-014,Wilson Inc Data Migration,Cloud Migration,Yellow,At-Risk,Testing,2026-02-24,2026-10-19,11,Mary Buckley,Ashley Peterson,63,High,4,88,22,0.0,5.0
4,DEAL-1005,CUST-005,Hamilton LLC ERP Integration,CRM Implementation,Green,Healthy,Build,2026-03-14,2026-11-18,4,Carolyn Carter,Kevin Ingram,83,High,2,48,11,0.0,8.0


In [45]:
project_risk_df["risk_score"] = (
    project_risk_df["ticket_count"] * 1.5 +
    project_risk_df["escalation_count"] * 15 +
    project_risk_df["risk_count"] * 5 +
    project_risk_df["estimated_delay_days"] * 1.2 +
    project_risk_df["open_blockers"] * 8 +
    (100 - project_risk_df["health_score"]) * 1.3
)

In [46]:
project_risk_df[
    [
        "deal_id",
        "project_name",
        "ticket_count",
        "escalation_count",
        "risk_count",
        "health_score",
        "estimated_delay_days",
        "risk_score"
    ]
].head()

,deal_id,project_name,ticket_count,escalation_count,risk_count,health_score,estimated_delay_days,risk_score
0,DEAL-1001,Brown-Porter API Modernization,28,0.0,2.0,78,14,121.4
1,DEAL-1002,Crosby-Rowland ERP Integration,12,0.0,5.0,85,4,67.3
2,DEAL-1003,Sanchez LLC CRM Implementation,45,4.0,3.0,22,57,400.3
3,DEAL-1004,Wilson Inc Data Migration,22,0.0,5.0,63,11,151.3
4,DEAL-1005,Hamilton LLC ERP Integration,11,0.0,8.0,83,4,99.4


In [47]:
top_risks = project_risk_df.sort_values(
    by="risk_score",
    ascending=False
)

top_risks[
    [
        "deal_id",
        "project_name",
        "project_status",
        "ticket_count",
        "escalation_count",
        "risk_count",
        "estimated_delay_days",
        "risk_score"
    ]
].head(10)

,deal_id,project_name,project_status,ticket_count,escalation_count,risk_count,estimated_delay_days,risk_score
68,DEAL-1069,Carpenter Group Workflow Automation,Red,54,13.0,4.0,28,544.5
91,DEAL-1092,"Banks, Fields and Mcconnell API Modernization",Red,66,13.0,4.0,29,527.1
81,DEAL-1082,Cross LLC Data Migration,Red,68,11.0,2.0,40,520.2
161,DEAL-1162,Nguyen LLC Cloud Migration,Red,63,12.0,5.0,32,504.3
18,DEAL-1019,"Gray, Williams and Hester Workflow Automation",Red,47,9.0,5.0,60,501.4
103,DEAL-1104,Bryant-Moreno Data Migration,Red,59,11.0,3.0,47,495.2
67,DEAL-1068,"Castro, King and Martin API Modernization",Red,40,11.0,4.0,59,494.8
127,DEAL-1128,Bryant-Moreno API Modernization,Red,56,11.0,5.0,54,492.9
197,DEAL-1198,"Weaver, Bryant and Ramirez CRM Implementation",Red,53,8.0,2.0,55,489.1
140,DEAL-1141,Shannon-Nichols API Modernization,Red,59,9.0,4.0,44,487.0


In [48]:
def categorize_risk(score):
    if score >= 180:
        return "Critical"
    elif score >= 120:
        return "High"
    elif score >= 70:
        return "Medium"
    else:
        return "Low"

project_risk_df["risk_level"] = (
    project_risk_df["risk_score"]
    .apply(categorize_risk)
)

In [49]:
project_risk_df[
    [
        "deal_id",
        "project_name",
        "risk_score",
        "risk_level"
    ]
].head(10)

,deal_id,project_name,risk_score,risk_level
0,DEAL-1001,Brown-Porter API Modernization,121.4,High
1,DEAL-1002,Crosby-Rowland ERP Integration,67.3,Low
2,DEAL-1003,Sanchez LLC CRM Implementation,400.3,Critical
3,DEAL-1004,Wilson Inc Data Migration,151.3,High
4,DEAL-1005,Hamilton LLC ERP Integration,99.4,Medium
5,DEAL-1006,Cross LLC Cloud Migration,426.8,Critical
6,DEAL-1007,May-Watkins Workflow Automation,69.3,Low
7,DEAL-1008,Mueller-Cohen CRM Implementation,166.9,High
8,DEAL-1009,Stewart PLC API Modernization,45.6,Low
9,DEAL-1010,Rogers-Padilla CRM Implementation,63.2,Low


In [50]:
top_project = top_risks.iloc[0]

top_project

deal_id                                             DEAL-1069
customer_id                                          CUST-015
project_name              Carpenter Group Workflow Automation
project_type                                  Cloud Migration
project_status                                            Red
health_profile                                       Critical
implementation_phase                                      UAT
kickoff_date                                       2026-03-12
planned_go_live                                    2026-09-22
estimated_delay_days                                       28
project_manager                              Catherine Taylor
technical_lead                                  Nicholas Hart
health_score                                               27
executive_visibility                                     High
open_blockers                                              15
budget_utilization_pct                                     66
ticket_c

In [51]:
deal_id = top_project["deal_id"]

project_docs = []

project_docs.extend(
    tickets[tickets["deal_id"] == deal_id]["combined_text"].tolist()
)

project_docs.extend(
    meeting_notes[
        meeting_notes["deal_id"] == deal_id
    ]["combined_text"].tolist()
)

project_docs.extend(
    escalations[
        escalations["deal_id"] == deal_id
    ]["combined_text"].tolist()
)

project_docs.extend(
    risks[
        risks["deal_id"] == deal_id
    ]["combined_text"].tolist()
)

In [52]:
context = "\n\n".join(project_docs[:20])

print(context[:3000])

Summary: Pending security approvals blocking deployment
Description: Security review approvals remain pending, impacting deployment timelines.
Priority: Critical
Status: Open
Category: Security Review

Summary: Historical data migration validation failures identified
Description: Migration validation revealed missing historical records and transformation inconsistencies.
Priority: Critical
Status: Pending
Category: Data Migration

Summary: Reporting discrepancies found during validation
Description: Customer reports do not match expected output due to transformation logic discrepancies.
Priority: Critical
Status: Pending
Category: Reporting Mismatch

Summary: API synchronization failing during nightly processing
Description: API jobs are intermittently failing due to token validation and timeout handling issues.
Priority: High
Status: In Progress
Category: API Integration

Summary: Pending security approvals blocking deployment
Description: Security review approvals remain pending, imp

In [ ]:
# AI Executive Analysis



In [54]:
prompt = f"""
You are an enterprise implementation risk advisor.

Analyze the following enterprise implementation project.

PROJECT NAME:
{top_project["project_name"]}

PROJECT STATUS:
{top_project["project_status"]}

RISK SCORE:
{top_project["risk_score"]}

ENTERPRISE CONTEXT:
{context}

Provide a detailed executive analysis including:

1. Executive Summary
2. Key Operational Risks
3. Escalation Concerns
4. Delivery Risk Assessment
5. Root Cause Analysis
6. Recommended Executive Actions
7. Overall Risk Severity
"""

In [55]:
response = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a senior enterprise implementation risk advisor."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3
)

risk_analysis = response.choices[0].message.content

print(risk_analysis)

**Executive Analysis: Carpenter Group Workflow Automation Project**

---

### 1. Executive Summary

The Carpenter Group Workflow Automation project is currently in a critical state, reflected by a Red status and a high risk score of 544.5. Multiple critical and high-priority issues spanning security, data migration, reporting, API integration, authentication, environment provisioning, workflow execution, and testing defects are causing significant delays and blocking deployment. The project is facing systemic challenges that threaten timely delivery, quality, and operational stability. Immediate executive intervention is required to address these risks, unblock dependencies, and realign the project trajectory.

---

### 2. Key Operational Risks

- **Security Review Delays:** Pending security approvals are repeatedly blocking deployment, indicating unresolved compliance or risk concerns that must be addressed to proceed.
- **Data Migration Failures:** Critical validation failures with m

In [56]:
project_risk_df.to_csv(
    "../data/project_risk_intelligence.csv",
    index=False
)